2025_05 【必須スキル】クレンジング

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
print(os.getcwd())

/app/src


In [3]:
# データの読み込み
# クレンジング対象の求人情報データ（id・企業名・スキル）
jobs_df = pd.read_csv('../data/cleansing_data.csv')

# スキルマッピング表
mapping_df = pd.read_csv('../data/key_mappingu.csv')

In [4]:
# データの確認
print(jobs_df.head(10))

   id       company_name                                    required_skills
0   0        株式会社シンフォニード  求める人材: \n＜必須条件＞\n下記項目を作業及びPMした経験（いずれも1年以上）\n\n...
1   1   Indeed Japan株式会社                                                NaN
2   2        株式会社シンフォニード  求める人材: \n＜必須条件＞\nデータ分析（アプリまたはWEBサービス）\n・SQLを用い...
3   3        株式会社ヘルスベイシス  求める人材: \n＜スキル＞\n・Pythonの実務利用経験\n・SQLの実務利用経験\n求...
4   4        株式会社シンフォニード  求める人材: \n＜必須条件＞\n?購買データや会員データ、Web行動データなどを用いたデー...
5   5  株式会社クリーク・アンド・リバー社  求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...
6   6          株式会社エイジレス  求める人材: \n＜必須要件＞\n・コミュニケーション能力および論理的思考\n・標準SQLで...
7   7        株式会社シンフォニード  求める人材: \n＜必須条件＞\n・データ分析によってビジネス上の課題を解決した経験（直近含...
8   8           バリュー株式会社  求める人材: \n＜以下のすべてを満たす方＞\n・データ／テクノロジーを駆使した企業変革、社...
9   9  株式会社クリーク・アンド・リバー社  求める人材: \n【必要要件】\n▼以下いずれかの経験を有する方\n・データマネジメントに関...


In [5]:
# データの確認
print(mapping_df.head(10))

   ID            raw_skill standard_skill
0   1                Adobe          Adobe
1   2              ANDROID        Android
2   3              Android        Android
3   4               APACHE         Apache
4   5               Apache         Apache
5   6                AVAYA          Avaya
6   7                Avaya          Avaya
7   8                  AWS            AWS
8   9  Amazon Web Services            AWS
9  10                AZURE          Azure


In [6]:
# マッチング結果の初期化
job_skill_pairs = []       # 中間テーブル用のデータ
unmatched_skills = set()   # マッチしなかったスキル文

# スキルの抽出に使用する正規表現
import re
skill_pattern = re.compile(r"・([^\n]+)")

In [7]:
# スキル項目のクレンジング
for idx, row in jobs_df.iterrows():
    job_id = row['id']
    company = row['company_name']

    # 改行や不要な空白を削除
    skill_text = str(row['required_skills']).replace('\n', ' ').strip().lower()

    # 正規表現でスキルを抽出
    skills = skill_pattern.findall(skill_text)

    # 抽出したスキルをマッチング処理
    matched_skills = set()

    for idx_mappling, mapping_row in mapping_df.iterrows():
        variant = str(mapping_row['raw_skill']).lower() # 小文字化
        normalized = mapping_row['standard_skill'] # 正規化されたスキル

        for skill in skills:
            if variant in skill:
                matched_skills.add(normalized)
    
    # マッチしたスキルがあれば、job_skill_parisに追加
    if matched_skills:
        for skill in matched_skills:
            job_skill_pairs.append({
                'job_id': job_id,
                'company_name': company,
                'skill': skill
            })
    else:
        unmatched_skills.add(skill_text)

In [8]:
# クレンジングの確認(マッチOK)
for item in job_skill_pairs[:10]:
    print(item)

{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'SQL'}
{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'Python'}
{'job_id': 0, 'company_name': '株式会社シンフォニード', 'skill': 'R'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'SQL'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'Python'}
{'job_id': 2, 'company_name': '株式会社シンフォニード', 'skill': 'R'}
{'job_id': 3, 'company_name': '株式会社ヘルスベイシス', 'skill': 'SQL'}
{'job_id': 3, 'company_name': '株式会社ヘルスベイシス', 'skill': 'Python'}
{'job_id': 4, 'company_name': '株式会社シンフォニード', 'skill': 'Looker'}
{'job_id': 4, 'company_name': '株式会社シンフォニード', 'skill': 'Tableau'}


In [9]:
# クレンジングの確認(マッチNO)
for text in list(unmatched_skills)[:10]:
    print(text)

求めている人材 ◎学歴不問 ◎分野を問わずデータ分析・運用経験をお持ちの方 （経験年数は問いません）  ・東京、首都圏の案件多数！ ・フルリモートも可能！ ・データアナリスト、データサイエンティストを目指したい方大歓迎  ※平均年齢36.2才  ★下記にひとつでも当てはまる方は大歓迎！ ・自分で開発案件を選びたい方。やりたい案件が決まっている方 ・自分が好きな言語・業界・技術を選んで開発の仕事をしたい方 ・会社のためよりお客様のために開発をしたい方 ・開発以外の雑務に時間を持って行かれたくない方 ・実力に応じた給与が欲しい方 ・プライベートも充実させたい方 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  【must】 ▼以下の条件の1つ以上に対し、3年以上の職務経験有すること。あるいは、以下の条件のいずれかに対し1年以上の職務経験およびwant条件を任意に数個を満たしていること。 ・データ分析・ai活用（上記職務内容の定義に従う）に関する自社及び他社（ベンダーの場合）の実業務システムの企画を中心的に主導して遂行した経験 ・データ分析・ai活用に関する自社及び他社の実業務システムをプロジェクトマネージャとして開発した経験 ・データ分析・ai活用に関する自社及び他社の実業務システムの運用に関し、特にデ 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  《 it業界デビューの方も大歓迎！ 》  ◆未経験者・第二新卒歓迎 ◆チームワークを大切にできる方 ◆人と話すことが好きな方 ◆it業界に興味がある方 ◆何かを作り出すことが好きな方 ◆正社員として地に足を付けたい方 ◆ワークライフバランスを大切にしたい方 求めている人材の情報量は適切ですか？ 十分 不足
求める人材:  35歳まで(例外事由3号イ:長期キャリア形成を図るため) ＜⭐️学歴不問・経験不問⭐️＞ 文系出身の人も多く、前職も営業や事務、企画、教師、アパレル販売、 携帯販売、飲食スタッフ、美容師、ホテル従業員、エステティシャン、エンジニアなどさまざまです。  業界・職種経験や特別な知識は一切必要ありません。 意欲や志望動機を重視した採用です。  第二新卒やフリーターの方、高卒の方も応募可！ ご興味があればまずはエントリーください。  【選考のポイント】 ≪こんな方は特に歓迎します

In [10]:
# DBにインポートするため、DataFrameに変換

# job_skill_pairsをDataFrame化
job_skill_df = pd.DataFrame(job_skill_pairs)

# 重複排除して正規化スキルの一覧を作成
unique_skills = job_skill_df['skill'].drop_duplicates().reset_index(drop=True)

# skill_idを付与して、skillsテーブル（DataFrame）を作成する
skills_df = pd.DataFrame({
    'skill_id': unique_skills.index + 1,
    'keyword': unique_skills
})

In [11]:
# DataFreame化した後の確認
print("----- {} の最初の10行 -----".format("skills_df"))
print(skills_df.head(10))

print("\n----- {} の最初の10行 -----".format("job_skill_df"))
print(job_skill_df.head(10))

----- skills_df の最初の10行 -----
   skill_id   keyword
0         1       SQL
1         2    Python
2         3         R
3         4    Looker
4         5   Tableau
5         6     BIツール
6         7        Go
7         8  BigQuery
8         9         C
9        10       AWS

----- job_skill_df の最初の10行 -----
   job_id company_name    skill
0       0  株式会社シンフォニード      SQL
1       0  株式会社シンフォニード   Python
2       0  株式会社シンフォニード        R
3       2  株式会社シンフォニード      SQL
4       2  株式会社シンフォニード   Python
5       2  株式会社シンフォニード        R
6       3  株式会社ヘルスベイシス      SQL
7       3  株式会社ヘルスベイシス   Python
8       4  株式会社シンフォニード   Looker
9       4  株式会社シンフォニード  Tableau


In [12]:
# csv出力
#skills_df.to_csv("../data/skills.csv", index=False)
#job_skill_df.to_csv("../data/job_posting_skills.csv", index=False)

【必須スキル】クレンジング後の整形

In [13]:
# クレンジングデータを読込み
pairs_df = pd.read_csv("../data/job_posting_skills.csv")

# スキルを一意にする
unique_skills = pairs_df['skill'].drop_duplicates().reset_index(drop=True)

# 各スキルにIDを付与する
skills_df = pd.DataFrame({
    'id': range(1, len(unique_skills)+ 1),
    'keyword': unique_skills
})
skills_df.to_csv("../data/skills.csv", index=False)

In [14]:
# job_idとskills_idの組み合わせで構成される中間テーブルを作成する
# skills.csvを読み込み
skills_df = pd.read_csv("../data/skills.csv")
pairs_df = pd.read_csv("../data/job_posting_skills.csv")

# スキル名でIDをマージする
merged_df = pairs_df.merge(skills_df, how='left', left_on='skill', right_on='keyword')

# 必要なカラムを選択してcsv出力する
final_df = merged_df[['job_id', 'id']].rename(columns={'id': 'skill_id'})
final_df.to_csv("../data/job_posting_skills_final.csv", index=False)